# FastRelax

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/uw-ipd/tmol/blob/kdidi/sphinx-docs/docs/tutorial/06_fast_relax.ipynb)

> **Prerequisite.** Complete [Packing and a Small Mutation Scan](04_packing_and_mutation_scan.ipynb) and [Minimization, constraints, and kinematics](05_minimization_constraints_kinematics.ipynb). FastRelax composes those packing and minimization primitives.

## Goals

This tutorial runs a deliberately small FastRelax example and explains the protocol rather than presenting it as an exact Rosetta port. You will:

- build the required `PackerPalette`, move map, and `FoldForest`;
- run one repeat on a checked-in 1UBQ slice;
- inspect the `fa_rep` and coordinate-constraint ramp;
- distinguish each stage's optimization weight from post-hoc full-weight rescoring;
- compare weighted score components and structures before and after relaxation;
- relax a four-member constrained ensemble in one GPU batch and click through every result; and
- see the verified adapter signature for optional kinematic minimization.

## Setup

In [ ]:
try:
    import google.colab  # noqa: F401
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True

if IN_COLAB:
    from urllib.request import urlopen

    exec(
        urlopen(
            "https://raw.githubusercontent.com/uw-ipd/tmol/"
            "kdidi/sphinx-docs/docs/tutorial/colab_setup.py"
        ).read(),
        globals(),
    )
    setup_colab(["tmol/tests/data/cif/1UBQ.cif"])

In [ ]:
from contextlib import redirect_stderr, redirect_stdout
from io import StringIO
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from biotite.structure.io import load_structure

import tmol
from tmol import beta2016_score_function
from tmol.io.pose_stack_from_biotite import pose_stack_from_biotite
from tmol.kinematics.fold_forest import FoldForest
from tmol.kinematics.move_map import CartesianMoveMap, MoveMap
from tmol.optimization.minimizers import run_cart_min, run_kin_min
from tmol.pack.packer_task import PackerPalette
from tmol.pose.pose_stack_builder import PoseStackBuilder
from tmol.relax.fast_relax import fast_relax
from tmol.score.constraint.utility import create_mainchain_coordinate_constraints
from tmol.score.score_types import ScoreType

SEED = 20260807
np.random.seed(SEED)
torch.manual_seed(SEED)
warnings.filterwarnings(
    "ignore", message=r"Sparse invariant checks are implicitly disabled.*"
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Deliberately short and shared by every minimizer in this tutorial, including
# the GPU batch. Production work should establish convergence for its use case.
TUTORIAL_MAX_ITER = 10


def show_table(frame):
    try:
        from itables import show
    except ImportError:
        return display(frame)
    return show(frame)


def score_components_by_pose(pose_stack, score_function):
    scorer = score_function.render_whole_pose_scoring_module(pose_stack)
    weighted_terms = scorer(
        pose_stack.coords, sum_terms=False, apply_weights=True
    )
    score_types = score_function.all_score_types()
    constraint_index = next(
        index
        for index, score_type in enumerate(score_types)
        if score_type == ScoreType.constraint
    )
    constraint_scores = weighted_terms[constraint_index]
    non_constraint_scores = weighted_terms.sum(dim=0) - constraint_scores
    total_scores = weighted_terms.sum(dim=0)
    return [
        {
            "non_constraint_score_units": float(non_constraint_scores[i].detach().cpu()),
            "constraint_score_units": float(constraint_scores[i].detach().cpu()),
            "reported_total_score_units": float(total_scores[i].detach().cpu()),
        }
        for i in range(pose_stack.n_poses)
    ]


def score_components(pose_stack, score_function):
    if pose_stack.n_poses != 1:
        raise ValueError("Use score_components_by_pose() for batched PoseStacks.")
    return score_components_by_pose(pose_stack, score_function)[0]

## Protocol and ramp schedule

TMol's `fast_relax(pose_stack, sfxn, packer_pallete, move_map, fold_forest, *, ...)` alternates rotamer packing and minimization. Each schedule entry independently scales the starting `fa_rep` weight for packing and minimization and can scale the starting coordinate-constraint weight. Explicit `cst_frac` values in dictionary entries define the ramp directly; automatic constraint ramping matters for schedule entries that omit them. The final minimization fraction should be `1.0` so the accepted structure is evaluated with full repulsion.

The production default is a four-step, two-repeat MonomerRelax2019-derived schedule. This tutorial uses two steps, one repeat, and `TUTORIAL_MAX_ITER=10` for every Cartesian or kinematic minimization to keep the checked-in example and CPU smoke test small. This is a runtime setting, not a claim of convergence; the optimizer's normal default is 200 iterations.

TMol's default `min_fn=None` selects Cartesian minimization, so the workflow passes a `CartesianMoveMap`. The `FoldForest` remains a required protocol argument but is ignored by the Cartesian adapter; it becomes active for the optional kinematic adapter below.

In [ ]:
repo_root = Path.cwd()
if not (repo_root / "tmol/tests/data/cif/1UBQ.cif").exists():
    repo_root = Path(tmol.__file__).resolve().parents[1]
cif_path = repo_root / "tmol" / "tests" / "data" / "cif" / "1UBQ.cif"
atom_array = load_structure(str(cif_path), model=1, include_bonds=True)
protein_slice = atom_array[(atom_array.chain_id == "A") & (atom_array.res_id <= 6)]
setup_diagnostics = StringIO()
try:
    with redirect_stdout(setup_diagnostics), redirect_stderr(setup_diagnostics):
        pose = pose_stack_from_biotite(protein_slice, device, no_optH=True)
        relax_start = create_mainchain_coordinate_constraints(pose)
except Exception:
    print(setup_diagnostics.getvalue())
    raise

score_function = beta2016_score_function(device)
score_function.set_weight(ScoreType.constraint, 1.0)
fa_rep_start = float(score_function.get_weight(ScoreType.fa_ljrep))
constraint_start = float(score_function.get_weight(ScoreType.constraint))

palette = PackerPalette()
cartesian_move_map = CartesianMoveMap()
fold_forest = FoldForest.reasonable_fold_forest(relax_start)
tiny_schedule = [
    {"fa_rep_pack_frac": 0.10, "fa_rep_min_frac": 0.20, "cst_frac": 1.0},
    {"fa_rep_pack_frac": 1.00, "fa_rep_min_frac": 1.00, "cst_frac": 0.0},
]

schedule_table = pd.DataFrame(tiny_schedule)
schedule_table["fa_rep_pack_weight"] = (
    schedule_table["fa_rep_pack_frac"] * fa_rep_start
)
schedule_table["fa_rep_min_weight"] = (
    schedule_table["fa_rep_min_frac"] * fa_rep_start
)
schedule_table["constraint_weight"] = schedule_table["cst_frac"] * constraint_start
print("input:", cif_path.name)
show_table(schedule_table)

## Run one repeat

The schedule table should show low repulsion in the first stage and full repulsion in the final stage, while the coordinate-constraint weight ramps to zero. If the constraint column is zero in every row, check both the score-function constraint weight and the pose's attached `ConstraintSet`.

The call below uses the complete current signature: pose, score function, palette, move map, and fold forest are positional; protocol controls are keyword-only. Coordinate constraints are already attached to `relax_start`, and the score function has a non-zero constraint weight, so the explicit `cst_frac` values in the schedule are active. The custom adapter keeps `max_iter` equal to the tutorial-wide value and records the constraint weight actually present during each minimization stage.

In [ ]:
def tutorial_cart_min(
    pose_stack, stage_score_function, *, fold_forest, move_map, verbose
):
    del fold_forest
    coord_mask = move_map.coord_mask if isinstance(move_map, CartesianMoveMap) else None
    return run_cart_min(
        pose_stack,
        stage_score_function,
        coord_mask=coord_mask,
        verbose=verbose,
        optimizer_kwargs={
            "max_iter": TUTORIAL_MAX_ITER,
            "verbose": verbose,
        },
    )


relax_stage_records = []


def recording_cart_min(
    pose_stack, stage_score_function, *, fold_forest, move_map, verbose
):
    constraint_weight = float(
        stage_score_function.get_weight(ScoreType.constraint)
    )
    minimized = tutorial_cart_min(
        pose_stack,
        stage_score_function,
        fold_forest=fold_forest,
        move_map=move_map,
        verbose=verbose,
    )
    relax_stage_records.append(
        {
            "pose": minimized.clone(),
            "constraint_weight_at_optimization": constraint_weight,
        }
    )
    return minimized


print(f"tutorial optimizer max_iter: {TUTORIAL_MAX_ITER}")
before_components = score_components(relax_start, score_function)
relaxed = fast_relax(
    relax_start,
    score_function,
    palette,
    cartesian_move_map,
    fold_forest,
    num_repeats=1,
    schedule=tiny_schedule,
    ramp_constraints=True,
    min_fn=recording_cart_min,
    verbose=False,
)
after_components = score_components(relaxed, score_function)

score_frame = pd.DataFrame(
    [
        {"structure": "before", **before_components},
        {"structure": "after", **after_components},
    ]
)
score_frame["non_constraint_score_change_units"] = (
    score_frame["non_constraint_score_units"]
    - before_components["non_constraint_score_units"]
)
show_table(score_frame)

In [ ]:
trajectory_structures = {"input": relax_start}
stage_constraint_weights = {"input": np.nan}
for stage_index, record in enumerate(relax_stage_records):
    label = f"stage {stage_index + 1}"
    trajectory_structures[label] = record["pose"]
    stage_constraint_weights[label] = record[
        "constraint_weight_at_optimization"
    ]
trajectory_structures["accepted final"] = relaxed
stage_constraint_weights["accepted final"] = np.nan

# fast_relax restores the starting constraint weight before returning. These
# scores are therefore a common post-hoc decomposition, not each stage's
# historical optimization objective.
posthoc_constraint_weight = float(
    score_function.get_weight(ScoreType.constraint)
)
trajectory_rows = []
trajectory_notes = {}
for frame_index, (label, frame_pose) in enumerate(trajectory_structures.items()):
    components = score_components(frame_pose, score_function)
    trajectory_rows.append(
        {
            "frame": frame_index,
            "label": label,
            "constraint_weight_at_optimization": stage_constraint_weights[label],
            "constraint_weight_for_posthoc_rescore": posthoc_constraint_weight,
            **components,
        }
    )
    optimized_weight = stage_constraint_weights[label]
    optimized_text = (
        "not an optimization endpoint"
        if np.isnan(optimized_weight)
        else f"optimized cst weight {optimized_weight:.3f}"
    )
    trajectory_notes[label] = (
        f"{optimized_text}; post-hoc non-constraint "
        f"{components['non_constraint_score_units']:.3f} score units; "
        f"constraint {components['constraint_score_units']:.3f} score units"
    )
trajectory_frame = pd.DataFrame(trajectory_rows)
show_table(trajectory_frame)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(
    trajectory_frame["frame"],
    trajectory_frame["non_constraint_score_units"],
    marker="o",
    label="post-hoc non-constraint score",
)
ax.plot(
    trajectory_frame["frame"],
    trajectory_frame["constraint_score_units"],
    marker="o",
    label="post-hoc constraint score",
)
ax.set(
    xticks=trajectory_frame["frame"],
    xticklabels=trajectory_frame["label"],
    ylabel="weighted score units",
    title="FastRelax endpoints rescored at restored constraint weight",
)
ax.tick_params(axis="x", rotation=20)
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

display(
    tmol.switchable_view(
        trajectory_structures,
        notes=trajectory_notes,
    )
)

**Expected observations.** The non-constraint and constraint components must be finite. “Non-constraint” is the exact description: it includes every weighted beta2016 term except the coordinate-constraint term, including empirical/statistical contributions. All reported values are weighted TMol score units, not kcal/mol.

`fast_relax()` restores the starting constraint weight before returning. The table records the weight present during each stage's optimization separately from the restored weight used for the common post-hoc score decomposition. In particular, the final stage is optimized with `cst_frac=0`, but its displayed constraint penalty is recalculated afterward at full starting weight; it diagnoses restraint violation and is not part of that stage's optimization objective. The structure switcher records minimization endpoints for each pack/min stage and the final accept-to-best result; it does not contain LBFGS line-search evaluations or internal packing trajectories.

A tiny pedagogical schedule is not guaranteed to improve every input. Large distortions, NaNs, or a missing ligand/residue are failures, not expected stochastic variation.

## Optional kinematic minimizer

`fast_relax` calls a custom minimizer as `min_fn(pose_stack, sfxn, *, fold_forest, move_map, verbose)`. The adapter below uses the verified `run_kin_min(pose_stack, sfxn, ff, mm, ...)` signature and the same tutorial iteration limit. To use it, supply a `MoveMap` (not a `CartesianMoveMap`) and pass `min_fn=kinematic_min_fn` to `fast_relax`.

This adapter is shown but not run in the CPU documentation build because it would duplicate the full packing schedule with a different minimizer. Notebook 05 executes both Cartesian and kinematic minimization directly; use that comparison before enabling the more expensive FastRelax variant here.

In [ ]:
def kinematic_min_fn(
    pose_stack, score_function, *, fold_forest, move_map, verbose
):
    return run_kin_min(
        pose_stack,
        score_function,
        fold_forest,
        move_map,
        optimizer_kwargs={
            "max_iter": TUTORIAL_MAX_ITER,
            "verbose": verbose,
        },
        verbose=verbose,
    )


kinematic_move_map = MoveMap.from_pose_stack(relax_start)
kinematic_move_map.move_all_named_torsions = True
kinematic_move_map.move_all_jumps = False
kinematic_move_map.move_all_root_jumps = False
# Example extension (not run here):
# kin_relaxed = fast_relax(
#     relax_start, score_function, palette, kinematic_move_map, fold_forest,
#     num_repeats=1, schedule=tiny_schedule, min_fn=kinematic_min_fn,
# )

## Relax an ensemble in one GPU batch

The single-structure call above and the batched call below use the same constrained protocol: each perturbed input receives main-chain coordinate constraints, the ensemble score function starts with the same nonzero constraint weight, the same two-stage schedule is used, and every minimization uses `TUTORIAL_MAX_ITER`. Four reproducibly perturbed conformations are assembled into one `PoseStack`, relaxed together, scored with the same constraint/non-constraint decomposition, and split only for the labeled structure switcher. This exposes the batch dimension to GPU kernels; measure it against four serial calls on the target GPU before choosing a production batch size.

Constraints are constructed after perturbation, so each ensemble member is restrained to its own starting main-chain coordinates, exactly as the single structure is restrained to its own start. The cell is marked `gpu-only`: CPU documentation CI preserves the code and records a skip message, while the Colab button at the top opens a GPU runtime where it can be executed.

In [ ]:
#| tags: [gpu-only]
if device.type != "cuda":
    print("Skipped: open this notebook in a Colab GPU runtime to run batch relax.")
else:
    ensemble_members = {}
    for member_index, noise_scale in enumerate((0.00, 0.01, 0.02, 0.03)):
        member = pose.clone()
        generator = torch.Generator(device=device).manual_seed(SEED + member_index)
        noise = torch.randn(
            member.coords.shape,
            generator=generator,
            device=device,
            dtype=member.coords.dtype,
        )
        member.coords[member.real_atoms] += noise[member.real_atoms] * noise_scale
        ensemble_members[f"input {member_index}"] = member

    ensemble = PoseStackBuilder.from_poses(
        list(ensemble_members.values()), device
    )
    ensemble = create_mainchain_coordinate_constraints(ensemble)
    ensemble_fold_forest = FoldForest.reasonable_fold_forest(ensemble)
    ensemble_sfxn = beta2016_score_function(device)
    ensemble_sfxn.set_weight(ScoreType.constraint, constraint_start)
    ensemble_before = score_components_by_pose(ensemble, ensemble_sfxn)

    relaxed_ensemble = fast_relax(
        ensemble,
        ensemble_sfxn,
        PackerPalette(),
        CartesianMoveMap(),
        ensemble_fold_forest,
        num_repeats=1,
        schedule=tiny_schedule,
        ramp_constraints=True,
        min_fn=tutorial_cart_min,
        verbose=False,
    )
    ensemble_after = score_components_by_pose(
        relaxed_ensemble, ensemble_sfxn
    )

    ensemble_rows = []
    for member_index in range(ensemble.n_poses):
        ensemble_rows.extend(
            [
                {
                    "member": member_index,
                    "structure": "before",
                    **ensemble_before[member_index],
                },
                {
                    "member": member_index,
                    "structure": "after",
                    **ensemble_after[member_index],
                },
            ]
        )
    ensemble_frame = pd.DataFrame(ensemble_rows)
    show_table(ensemble_frame)

    comparison_structures = {}
    comparison_notes = {}
    for member_index in range(ensemble.n_poses):
        before_label = f"member {member_index} — before"
        after_label = f"member {member_index} — after"
        comparison_structures[before_label] = ensemble.split(member_index)
        comparison_structures[after_label] = relaxed_ensemble.split(member_index)
        before = ensemble_before[member_index]
        after = ensemble_after[member_index]
        comparison_notes[before_label] = (
            f"non-constraint {before['non_constraint_score_units']:.3f}; "
            f"constraint {before['constraint_score_units']:.3f} score units"
        )
        comparison_notes[after_label] = (
            f"non-constraint {after['non_constraint_score_units']:.3f}; "
            f"constraint {after['constraint_score_units']:.3f} score units"
        )
    display(
        tmol.switchable_view(
            comparison_structures,
            notes=comparison_notes,
        )
    )

## Rosetta comparison

Both protocols alternate side-chain repacking with minimization while ramping steric repulsion, and both can ramp coordinate constraints. A common Rosetta FastRelax workflow minimizes torsional degrees of freedom through a `MoveMap` unless Cartesian minimization is explicitly selected. TMol's current `fast_relax` default is Cartesian (`min_fn=None` selects its Cartesian adapter); torsional minimization requires the explicit callable and `MoveMap` shown above. This changes the active degrees of freedom and can change the resulting trajectory.

Rosetta FastRelax has a mature script language, MoveMap factories, symmetry and membrane integrations, and many protocol options. TMol implements a smaller tensor-native subset: a Python schedule, `PackerTask` operations, the Cartesian default plus an optional callable minimizer, and accept-to-best across repeats.

The algorithms and score functions are not numerically interchangeable. Treat TMol FastRelax as partial protocol parity for batched refinement, not as an exact port or a guarantee of reproducing Rosetta trajectories.

## Exercises

1. Replace the tiny schedule with `DEFAULT_RELAX_SCHEDULE` and compare runtime and score.
2. Keep constraints active in the final step and compare Cα displacement.
3. Pass `kinematic_min_fn` with the prepared `kinematic_move_map` and compare Cartesian and kinematic results.
4. Write a task operation that restricts packing to a selected block mask.
5. Batch two small poses and inspect accept-to-best behavior per pose.

## References

- [Rosetta Relax tutorial](https://docs.rosettacommons.org/demos/latest/tutorials/Relax_Tutorial/Relax)
- [Rosetta FastRelax implementation](https://github.com/RosettaCommons/rosetta/blob/main/source/src/protocols/relax/FastRelax.cc)
- [PyRosetta packing, design, and regional relax](https://github.com/RosettaCommons/PyRosetta.notebooks/blob/master/notebooks/06.02-Packing-design-and-regional-relax.ipynb)
- [PyRosetta refinement documentation](https://www.pyrosetta.org/documentation)